# DSN 2026 "Find the Language I Am Speaking": team godzilla solution notebook

Spoken language identification of 2–30 s clips into 13 labels (`yor ibo hau ego anw efi ibb idu igl nup tiv urh other`).
Scored submissions: **D5b, final 0.6222** (weighted F1 0.6951, efficiency 0.790), public leaderboard first place at the time
of writing. Later submissions (D5c, D7) are pending scoring.

**Model (one standard `HubertForSequenceClassification`, loads with `AutoModelForAudioClassification`, no custom code):**
- Backbone `ajesujoba/AfriHuBERT` (12 layers, 94.6M parameters, fp16 safetensors, 0.189 GB), `use_weighted_layer_sum=True`, mean pooling, frozen CNN feature encoder.
- Fine-tuned for 8,000 steps (batch 32, AdamW, OneCycle, lr 5e-5 body / 5e-4 head) on random 3–6 s crops with class-uniform, source-balanced (hours^0.5, external sources capped at half a class) and speaker-balanced (size^0.5) sampling.
- Audio augmentation on every training crop: random noise floor, speed/pitch 0.9–1.1, random 8-band EQ, synthetic reverb, coloured noise or babble, telephone channel (300–3400 Hz, 8 kHz, mu-law).
- Label cleaning: 32 competition clips whose audio is clearly English were relabelled `other`.
- Weight averaging: two full-data models trained from the same head initialisation with different data-order/augmentation seeds are averaged (still one model of the same size).
- Per-language decision offsets added to `classifier.bias`: tuned on speaker-held-out 3-fold cross-validation (coordinate ascent on a bootstrap of the public-test class mix and clean/noisy/phone conditions, shrunk ×0.75), then corrected using the confusion matrix of our previous public-leaderboard submission.
- Inference: the full clip, no custom code.

**Validation:** 3 speaker-grouped folds over train+validation (Nupe has no held-out competition speaker, so held-out VoiceAfrica speakers stand in), each fold scored on clean, noisy and phone-degraded copies and resampled to the public-test class mix; the exact score formula (`src/score.py`) is reproduced locally, efficiency included.


## Training data

- Competition data: [Data-Science-Nigeria/dsn-2026-lid-train](https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train) (train + validation; afrispeech_ipa CC BY 4.0, hypa_voices Apache 2.0, omnilingual_asr CC BY 4.0). No other use of the prohibited corpora.
- External, openly licensed datasets (all listed in every model card):

| Dataset | Config / split | Licence | Used as | URL |
|---|---|---|---|---|
| offiongbassey/efik_speech_dataset | default/train | CC-BY-4.0 | `efi` | https://huggingface.co/datasets/offiongbassey/efik_speech_dataset |
| google/fleurs | ar_eg/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | en_us/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | es_419/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | ff_sn/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | fr_fr/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | ha_ng/train | CC-BY-4.0 | `hau` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | ig_ng/train | CC-BY-4.0 | `ibo` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | lg_ug/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | ln_cd/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | pt_br/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | sw_ke/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | wo_sn/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/google/fleurs |
| google/fleurs | yo_ng/train | CC-BY-4.0 | `yor` | https://huggingface.co/datasets/google/fleurs |
| McGill-NLP/NaijaS2ST | default/dev+train language=english | CC-BY-4.0 | `other` | https://huggingface.co/datasets/McGill-NLP/NaijaS2ST |
| McGill-NLP/NaijaS2ST | default/train language=pidgin | CC-BY-4.0 | `other` | https://huggingface.co/datasets/McGill-NLP/NaijaS2ST |
| asr-nigerian-pidgin/nigerian-pidgin-1.0 | default/train | CC-BY-4.0 | `other` | https://huggingface.co/datasets/asr-nigerian-pidgin/nigerian-pidgin-1.0 |
| google/WaxalNLP | aka_asr/train | CC-BY-4.0 (University of Ghana) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| google/WaxalNLP | bau_tts/train | CC-BY-4.0 (University of Ghana) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| google/WaxalNLP | dag_asr/train | CC-BY-4.0 (University of Ghana) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| google/WaxalNLP | ewe_asr/train | CC-BY-4.0 (University of Ghana) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| google/WaxalNLP | ful_asr/train | CC-BY-SA-4.0 (Digital Umuganda) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| google/WaxalNLP | pcm_tts/train | CC-BY-SA-4.0 (Media Trust) | `other` | https://huggingface.co/datasets/google/WaxalNLP |
| naijavoices/voiceafrica-datasets | igl/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `igl` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | nup/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `nup` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | urh/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `urh` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | knc/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | bcz/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | ble/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | bvb/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | fan/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | krx/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | pov/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| naijavoices/voiceafrica-datasets | srr/train | CC-BY-NC-SA-4.0 + NaijaVoices terms (https://naijavoices.com/terms-and-conditions); no voice cloning | `other` | https://huggingface.co/datasets/naijavoices/voiceafrica-datasets |
| Common Voice cv-corpus-27.0-2026-09-17 (ibb) | validated | CC0-1.0 | `ibb` | https://mozilladatacollective.com/datasets/cmu5uwjhw008znq07h4vu9kxp |
| Speech Data Collection for The Nupe Language (Mozilla Data Collective, 7 speakers) | all | CC-BY-NC-SA-4.0 | `nup` | https://mozilladatacollective.com/datasets/cmihkoth10246md07dtojxehg |
| ext_wikimedia (44 recordings, per-file attribution in the model card) | speech segments only | CC BY 4.0, CC BY-SA 4.0 | `anw`, `efi`, `ibb`, `other`, `urh` | https://commons.wikimedia.org / Bloom Library |
| ext_wikitongues (4 recordings, per-file attribution in the model card) | speech segments only | CC BY 4.0, CC BY-SA 4.0, CC0 | `ibb`, `other` | https://commons.wikimedia.org / Bloom Library |
| ext_bloom (2 recordings, per-file attribution in the model card) | speech segments only | CC BY 4.0 | `ego`, `tiv` | https://commons.wikimedia.org / Bloom Library |

Speaker-overlap check: every new source was compared with the competition speakers using WavLM x-vectors (`src/voice_overlap.py`); new speakers whose voice matched a competition or VoiceAfrica speaker were dropped (2 of 17 Common Voice Ibibio speakers, 2 of 7 MDC Nupe speakers). Downloaded copies are private and not re-shared.

## Reproduction

1. Put the competition parquet shards, the converted external shards and the fold files under `/kaggle/input/...` (private Kaggle datasets; the training script globs every `*.parquet` under `/kaggle/input` and the fold file named in `CONFIG`).
2. Run `src/train_kernel.py` on a T4 with the configurations below (about 50 min per full model).
3. Average the two full models, add the offsets, run the organisers' checker, upload to Hugging Face and write `submission.zip`.

The cells below write the exact source files used (also in the team repository) and then show the commands.


In [ ]:
%%writefile src/score.py
"""Local copy of the DSN 2026 LID competition score.

Final score = language-weighted F1 x (0.5 + 0.5 x efficiency), as described on the
competition's Evaluation page. Run `python src/score.py --selftest` to check it against
the organisers' worked example and a published leaderboard row.

Usage:
    python src/score.py preds.csv              # columns: label, pred (ISO codes)
    python src/score.py preds.csv --params-m 38 --repo-gb 0.08 --mem-gb 1.2 --latency-s 0.01
"""
import argparse
import csv
import math
import sys

LABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb", "idu", "igl", "nup", "tiv", "urh", "other"]
HIGH_RESOURCE = {"yor", "ibo", "hau"}
# 4.3% for high-resource classes and 8.7% for the rest, i.e. exactly 1/23 and 2/23.
WEIGHTS = {c: (1 if c in HIGH_RESOURCE else 2) / 23 for c in LABELS}

PARAM_LIMIT_M = 350.0
REPO_LIMIT_GB = 1.5
MEMORY_LIMIT_GB = 4.0
LATENCY_FLOOR_S = 0.01
LATENCY_LIMIT_S = 0.5
RANKING_F1_FLOOR = 0.60


def per_class_f1(y_true, y_pred):
    f1 = {}
    for c in LABELS:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        if tp + fp + fn == 0:
            f1[c] = None  # class absent from both truth and predictions
        else:
            f1[c] = 2 * tp / (2 * tp + fp + fn)
    return f1


def weighted_f1(y_true, y_pred):
    """Fixed-share F1. Classes absent from the evaluation set (e.g. nup in the official
    validation split) are dropped and the remaining shares renormalised, so local numbers
    stay comparable; the hidden test set contains every class."""
    f1 = per_class_f1(y_true, y_pred)
    present = [c for c in LABELS if f1[c] is not None]
    total = sum(WEIGHTS[c] for c in present)
    return sum(WEIGHTS[c] * f1[c] for c in present) / total, f1


def efficiency(params_m, repo_gb, mem_gb, latency_s):
    clip = lambda x: max(0.0, min(1.0, x))
    size = (clip(1 - params_m / PARAM_LIMIT_M) + clip(1 - repo_gb / REPO_LIMIT_GB)) / 2
    memory = clip(1 - mem_gb / MEMORY_LIMIT_GB)
    t = max(latency_s, LATENCY_FLOOR_S)
    speed = clip(1 - math.log(t / LATENCY_FLOOR_S) / math.log(LATENCY_LIMIT_S / LATENCY_FLOOR_S))
    return (size + memory + speed) / 3, {"size": size, "memory": memory, "speed": speed}


def final_score(wf1, eff):
    return wf1 * (0.5 + 0.5 * eff)


def within_limits(params_m, repo_gb, mem_gb, latency_s):
    return (params_m <= PARAM_LIMIT_M and repo_gb <= REPO_LIMIT_GB
            and mem_gb <= MEMORY_LIMIT_GB and latency_s <= LATENCY_LIMIT_S)


def selftest():
    # Organisers' example: F1 0.90, 100M params, 0.4 GB repo, 1 GB memory, 0.05 s -> about 0.76.
    eff, _ = efficiency(100, 0.4, 1.0, 0.05)
    assert abs(final_score(0.90, eff) - 0.76) < 0.005, final_score(0.90, eff)
    # Leaderboard row (2026-10-20): 20.2M params, 0.04 GB, 1.19 GB, 0.005 s, F1 0.549
    # -> size 0.958, memory 0.702, speed 1.0, efficiency 0.887, final 0.5179.
    eff, parts = efficiency(20.2, 0.04, 1.19, 0.005)
    assert abs(parts["size"] - 0.958) < 0.001 and abs(parts["memory"] - 0.702) < 0.001
    assert abs(eff - 0.887) < 0.001 and abs(final_score(0.549, eff) - 0.5179) < 0.0005
    # Weighted F1 on a perfect prediction is 1.
    assert abs(weighted_f1(LABELS, LABELS)[0] - 1.0) < 1e-12
    print("selftest passed")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("preds", nargs="?", help="CSV with columns label,pred")
    ap.add_argument("--params-m", type=float, help="parameters in millions")
    ap.add_argument("--repo-gb", type=float, help="repository size in GB")
    ap.add_argument("--mem-gb", type=float, help="peak memory in GB (max of RAM and GPU)")
    ap.add_argument("--latency-s", type=float, help="average seconds per clip")
    ap.add_argument("--selftest", action="store_true")
    args = ap.parse_args()

    if args.selftest:
        selftest()
        return
    if not args.preds:
        ap.error("give a predictions CSV or --selftest")

    with open(args.preds, newline="") as f:
        rows = list(csv.DictReader(f))
    y_true = [r["label"] for r in rows]
    y_pred = [r["pred"] for r in rows]
    unknown = sorted(set(y_true + y_pred) - set(LABELS))
    if unknown:
        sys.exit(f"unknown labels: {unknown}")

    wf1, f1 = weighted_f1(y_true, y_pred)
    print(f"clips: {len(rows)}")
    print(f"language-weighted F1: {wf1:.4f}" + ("" if wf1 >= RANKING_F1_FLOOR else "  (below the 0.60 ranking floor)"))
    for c in LABELS:
        n = sum(1 for t in y_true if t == c)
        print(f"  {c:<6} F1 {'  n/a' if f1[c] is None else f'{f1[c]:.3f}'}  (n={n}, share {WEIGHTS[c]:.1%})")

    eff_args = (args.params_m, args.repo_gb, args.mem_gb, args.latency_s)
    if all(a is not None for a in eff_args):
        eff, parts = efficiency(*eff_args)
        print(f"efficiency: {eff:.3f}  (size {parts['size']:.3f}, memory {parts['memory']:.3f}, speed {parts['speed']:.3f})")
        print(f"final score: {final_score(wf1, eff):.4f}")
        if not within_limits(*eff_args):
            print("WARNING: breaks a competition limit, so it would not be scored")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/make_folds.py
"""Build the 3 speaker-grouped CV folds (A, B, C) over pooled train + validation.

Groups:
- hypa_voices: speaker_id, pooled across labels (bilingual speakers stay in one fold)
- omnilingual_asr: speaker key string, pooled across anw/ego/idu
- afrispeech_ipa: no speaker ids. With --afri-clusters (id,cluster CSV from speaker
  embeddings) each cluster is a group; without it the organisers' split is used as a
  stand-in (validation -> A, train -> B/C by hash), which leaks speakers within
  afrispeech and is only good enough for the rung-0 smoke test.

The scarce-class speakers rotate as in the research report, so every fold holds out one
igl, one ibb, one efi and one hypa-tiv speaker group. nup has a single speaker
(SPK-147), who stays in training for every fold (fold "T").

Usage:
    python src/make_folds.py metadata.parquet out.csv [--afri-clusters clusters.csv]
"""
import argparse
import hashlib
from collections import defaultdict

import pandas as pd

ROTATION = {
    "A": ["SPK-059", "SPK-071", "SPK-011", "SPK-120", "SPK-041"],
    "B": ["SPK-086", "SPK-004", "SPK-040", "SPK-052", "SPK-061"],
    "C": ["SPK-031", "SPK-104", "SPK-140", "SPK-065", "SPK-129", "SPK-096"],
}
ALWAYS_TRAIN = ["SPK-147"]
FOLDS = ["A", "B", "C"]
SILENT_CLIPS = {"69cd0258ddc28532"}


def stable_fold(key, choices):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return choices[h % len(choices)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("metadata")
    ap.add_argument("out")
    ap.add_argument("--afri-clusters")
    args = ap.parse_args()

    m = pd.read_parquet(args.metadata, columns=["id", "split", "label", "source", "speaker_id", "speaker_key",
                                                "duration", "near_dup_of"])
    drop = m["id"].isin(SILENT_CLIPS) | (m["near_dup_of"].fillna("") != "")
    m = m[~drop].copy()

    m["group"] = m["speaker_key"]
    afri = m["source"] == "afrispeech_ipa"
    if args.afri_clusters:
        cl = pd.read_csv(args.afri_clusters, dtype=str).set_index("id")["cluster"]
        m.loc[afri, "group"] = "afrispeech_ipa:cl" + m.loc[afri, "id"].map(cl)
        assert m.loc[afri, "group"].notna().all(), "clusters missing for some afrispeech clips"
    else:
        m.loc[afri, "group"] = "afrispeech_ipa:" + m.loc[afri, "split"] + ":" + m.loc[afri, "id"]

    fold_of = {}
    for f, spks in ROTATION.items():
        for s in spks:
            fold_of[f"hypa_voices:{s}"] = f
    for s in ALWAYS_TRAIN:
        fold_of[f"hypa_voices:{s}"] = "T"

    if not args.afri_clusters:
        for g, split in m.loc[afri, ["group", "split"]].itertuples(index=False):
            fold_of[g] = "A" if split == "validation" else stable_fold(g, ["B", "C"])

    # Greedy stratified assignment of the remaining groups: each group goes to the fold
    # holding the fewest seconds of the group's main label so far.
    seconds = defaultdict(float)  # (label, fold) -> seconds
    for g, f in fold_of.items():
        for lab, dur in m.loc[m["group"] == g, ["label", "duration"]].itertuples(index=False):
            seconds[(lab, f)] += dur
    rest = m[~m["group"].isin(fold_of)]
    sizes = rest.groupby("group")["duration"].sum().sort_values(ascending=False)
    main_label = rest.groupby("group")["label"].agg(lambda s: s.value_counts().index[0])
    for g in sizes.index:
        lab = main_label[g]
        f = min(FOLDS, key=lambda x: (seconds[(lab, x)], x))
        fold_of[g] = f
        for l2, dur in rest.loc[rest["group"] == g, ["label", "duration"]].itertuples(index=False):
            seconds[(l2, f)] += dur

    m["fold"] = m["group"].map(fold_of)
    m[["id", "label", "source", "group", "fold"]].to_csv(args.out, index=False)

    table = m.pivot_table(index="label", columns="fold", values="duration", aggfunc="sum").fillna(0) / 3600
    speakers = m.pivot_table(index="label", columns="fold", values="group", aggfunc="nunique").fillna(0).astype(int)
    print(f"{len(m)} clips ({drop.sum()} dropped)\nhours per fold:\n{table.round(2)}\ngroups per fold:\n{speakers}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/convert_commonvoice.py
"""Convert one Common Voice locale (Mozilla Data Collective download) to the competition parquet schema.

Uses validated.tsv (plus other.tsv with --with-other), drops anything in invalidated.tsv, caps clips
per speaker so a few heavy contributors don't dominate, and keeps 2-30 s clips. Writes parquet
shards, a folds file (every clip fold X: always train, never evaluated) and a dataset-level manifest
row for the model card.

    python src/convert_commonvoice.py <locale_dir> <label> <out_dir> --release cv-corpus-27.0-2026-09-11
        [--per-speaker 120] [--with-other] [--url https://mozilladatacollective.com/datasets/...]
"""
import argparse
import hashlib
import io
import os
import random
import subprocess
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf

LABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb", "idu", "igl", "nup", "tiv", "urh", "other"]
SR, MAX_SHARD, ROWS_PER_RG = 16000, 450e6, 64
SCHEMA = pa.schema([("id", pa.string()), ("audio", pa.struct([("bytes", pa.binary()), ("path", pa.string())])),
                    ("label", pa.string()), ("label_id", pa.int64()), ("speaker_id", pa.string()),
                    ("source", pa.string()), ("duration", pa.float64())])


def read_tsv(path):
    if not os.path.exists(path):
        return pd.DataFrame(columns=["client_id", "path"])
    return pd.read_csv(path, sep="\t", quoting=3, dtype=str, on_bad_lines="skip")


def decode(path):
    p = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(SR),
                        "-f", "s16le", "-"], capture_output=True)
    if p.returncode != 0 or not p.stdout:
        return None
    x = np.frombuffer(p.stdout, np.int16)
    # trim the silent lead-in and tail Common Voice recordings usually have (frames < -50 dBFS)
    f = x[: len(x) // 400 * 400].reshape(-1, 400).astype(np.float32) / 32768
    loud = np.where(10 * np.log10((f ** 2).mean(1) + 1e-10) > -50)[0]
    if len(loud) == 0:
        return None
    a, b = max(0, loud[0] - 4) * 400, min(len(x), (loud[-1] + 5) * 400)
    return x[a:b]


def flac(x):
    buf = io.BytesIO()
    sf.write(buf, x, SR, format="FLAC", subtype="PCM_16")
    return buf.getvalue()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("locale_dir")
    ap.add_argument("label", choices=LABELS)
    ap.add_argument("out_dir")
    ap.add_argument("--release", required=True)
    ap.add_argument("--per-speaker", type=int, default=120)
    ap.add_argument("--with-other", action="store_true")
    ap.add_argument("--url", default="https://mozilladatacollective.com")
    a = ap.parse_args()
    os.makedirs(a.out_dir, exist_ok=True)
    src = f"ext_cv_{a.label}"

    rows = read_tsv(os.path.join(a.locale_dir, "validated.tsv"))
    if a.with_other:
        rows = pd.concat([rows, read_tsv(os.path.join(a.locale_dir, "other.tsv"))])
    bad = set(read_tsv(os.path.join(a.locale_dir, "invalidated.tsv"))["path"])
    rows = rows[~rows.path.isin(bad)].drop_duplicates("path")
    print(f"{len(rows)} usable clips from {rows.client_id.nunique()} speakers")
    rng = random.Random(2026)
    pick = []
    for _, g in rows.groupby("client_id"):
        idx = list(g.index) if len(g) <= a.per_speaker else rng.sample(list(g.index), a.per_speaker)
        pick.append(g.loc[idx])
    rows = pd.concat(pick).reset_index(drop=True)

    clips = os.path.join(a.locale_dir, "clips")
    with ThreadPoolExecutor(4) as ex:
        waves = list(ex.map(lambda p: decode(os.path.join(clips, p)), rows.path))
    recs = []
    for r, x in zip(rows.itertuples(), waves):
        if x is None or not 2.0 <= len(x) / SR <= 30.0:
            continue
        recs.append({"id": f"cv_{a.label}_{hashlib.sha1(r.path.encode()).hexdigest()[:16]}",
                     "audio": {"bytes": flac(x), "path": r.path}, "label": a.label, "label_id": LABELS.index(a.label),
                     "speaker_id": "cv_" + hashlib.sha1(r.client_id.encode()).hexdigest()[:12], "source": src,
                     "duration": round(len(x) / SR, 3)})
    rng.shuffle(recs)
    t = pa.Table.from_pylist(recs, schema=SCHEMA)
    per_shard = max(ROWS_PER_RG, int(MAX_SHARD / max(1, t.nbytes / max(1, t.num_rows))) // ROWS_PER_RG * ROWS_PER_RG)
    for k, s in enumerate(range(0, t.num_rows, per_shard)):
        pq.write_table(t.slice(s, per_shard), os.path.join(a.out_dir, f"{src}-{k:05d}.parquet"),
                       row_group_size=ROWS_PER_RG, compression="snappy")

    d = t.select(["id", "label", "speaker_id", "source", "duration"]).to_pandas()
    pd.DataFrame({"id": d.id, "label": d.label, "source": d.source, "group": d.speaker_id, "fold": "X"}).to_csv(
        os.path.join(a.out_dir, f"folds_{src}.csv"), index=False)
    pd.DataFrame([{"dataset_id": f"Common Voice {a.release} ({a.label})", "config_split": "validated" + (" + other" if a.with_other else ""),
                   "revision": a.release.replace("cv-corpus-", ""), "license": "CC0-1.0", "label": a.label, "url": a.url,
                   "clips": len(d), "speakers": d.speaker_id.nunique(), "hours": round(d.duration.sum() / 3600, 2)}]).to_csv(
        os.path.join(a.out_dir, "manifest.csv"), index=False)
    spk = d.groupby("speaker_id").duration.agg(["size", "sum"]).sort_values("sum", ascending=False)
    spk.to_csv(os.path.join(a.out_dir, "speakers.csv"))
    print(f"kept {len(d)} clips, {d.speaker_id.nunique()} speakers, {d.duration.sum() / 3600:.2f} h; "
          f"top speakers (clips, s):\n{spk.head(8).round(0).to_string()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/convert_audio_folder.py
"""Convert any folder of audio files (plus optional metadata table) to the competition parquet schema.

Speaker id comes from, in order: a metadata TSV/CSV column named like speaker/client_id/reader
(matched to the file name), else the first directory level under the root that contains more than one
file, else the whole set is one speaker. Clips longer than --max-s are cut into pieces; shorter than 2 s
are dropped. Silent lead-in and tail are trimmed. Writes parquet shards, folds_<source>.csv (fold X) and a
manifest row.

    python src/convert_audio_folder.py <root> <label> <source> <out_dir> --name "..." --license ... --url ...
        [--per-speaker 120] [--max-s 20]
"""
import argparse
import csv
import glob
import hashlib
import io
import os
import random
import subprocess
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf

LABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb", "idu", "igl", "nup", "tiv", "urh", "other"]
SR, MAX_SHARD, ROWS_PER_RG = 16000, 450e6, 64
AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm", ".aac")
SCHEMA = pa.schema([("id", pa.string()), ("audio", pa.struct([("bytes", pa.binary()), ("path", pa.string())])),
                    ("label", pa.string()), ("label_id", pa.int64()), ("speaker_id", pa.string()),
                    ("source", pa.string()), ("duration", pa.float64())])


def decode(path):
    p = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(SR),
                        "-f", "s16le", "-"], capture_output=True)
    if p.returncode != 0 or len(p.stdout) < 2 * SR:
        return None
    x = np.frombuffer(p.stdout, np.int16)
    f = x[: len(x) // 400 * 400].reshape(-1, 400).astype(np.float32) / 32768
    loud = np.where(10 * np.log10((f ** 2).mean(1) + 1e-10) > -50)[0]
    if len(loud) == 0:
        return None
    return x[max(0, loud[0] - 4) * 400: min(len(x), (loud[-1] + 5) * 400)]


def flac(x):
    buf = io.BytesIO()
    sf.write(buf, x, SR, format="FLAC", subtype="PCM_16")
    return buf.getvalue()


def speaker_map(root, files):
    """file path -> speaker id, from a metadata table if one names speakers, else from directories."""
    for t in glob.glob(f"{root}/**/*.tsv", recursive=True) + glob.glob(f"{root}/**/*.csv", recursive=True):
        try:
            df = pd.read_csv(t, sep="\t" if t.endswith(".tsv") else ",", dtype=str, quoting=3, on_bad_lines="skip")
        except Exception:
            continue
        spk_col = next((c for c in df.columns if c.lower() in ("client_id", "speaker_id", "speaker", "reader", "voice", "spk")), None)
        file_col = next((c for c in df.columns if c.lower() in ("path", "file", "filename", "file_name", "audio", "wav", "clip")), None)
        if spk_col and file_col:
            m = {os.path.basename(str(p)): s for p, s in zip(df[file_col], df[spk_col])}
            got = {f: m.get(os.path.basename(f)) for f in files}
            if sum(v is not None for v in got.values()) > len(files) / 2:
                print(f"speakers from {os.path.relpath(t, root)} column '{spk_col}': {len(set(v for v in got.values() if v))} speakers")
                return {f: (v or "unknown") for f, v in got.items()}
    rel = [os.path.relpath(f, root).split(os.sep) for f in files]
    for level in range(max(len(r) for r in rel) - 1):
        keys = {tuple(r[: level + 1]) for r in rel if len(r) > level + 1}
        if len(keys) > 1:
            print(f"speakers from directory level {level}: {len(keys)} groups")
            return {f: "/".join(r[: level + 1]) for f, r in zip(files, rel)}
    print("no speaker information: treating the whole set as one speaker")
    return {f: "all" for f in files}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("root")
    ap.add_argument("label", choices=LABELS)
    ap.add_argument("source")
    ap.add_argument("out_dir")
    ap.add_argument("--name", required=True)
    ap.add_argument("--license", required=True)
    ap.add_argument("--url", required=True)
    ap.add_argument("--per-speaker", type=int, default=120)
    ap.add_argument("--max-s", type=float, default=20.0)
    a = ap.parse_args()
    os.makedirs(a.out_dir, exist_ok=True)
    files = sorted(f for f in glob.glob(f"{a.root}/**/*", recursive=True) if f.lower().endswith(AUDIO_EXT))
    print(f"{len(files)} audio files")
    spk = speaker_map(a.root, files)
    rng = random.Random(2026)
    by = {}
    for f in files:
        by.setdefault(spk[f], []).append(f)
    pick = []
    for s, fs in by.items():
        pick += fs if len(fs) <= a.per_speaker else rng.sample(fs, a.per_speaker)
    with ThreadPoolExecutor(4) as ex:
        waves = list(ex.map(decode, pick))
    recs = []
    for f, x in zip(pick, waves):
        if x is None:
            continue
        pieces = [x[i: i + int(a.max_s * SR)] for i in range(0, len(x), int(a.max_s * SR))]
        for j, piece in enumerate(pieces):
            if len(piece) < 2 * SR:
                continue
            h = hashlib.sha1(f"{os.path.relpath(f, a.root)}#{j}".encode()).hexdigest()[:16]
            recs.append({"id": f"{a.source}_{h}", "audio": {"bytes": flac(piece), "path": os.path.basename(f)},
                         "label": a.label, "label_id": LABELS.index(a.label),
                         "speaker_id": f"{a.source}_" + hashlib.sha1(str(spk[f]).encode()).hexdigest()[:10],
                         "source": a.source, "duration": round(len(piece) / SR, 3)})
    rng.shuffle(recs)
    t = pa.Table.from_pylist(recs, schema=SCHEMA)
    per_shard = max(ROWS_PER_RG, int(MAX_SHARD / max(1, t.nbytes / max(1, t.num_rows))) // ROWS_PER_RG * ROWS_PER_RG)
    for k, s in enumerate(range(0, t.num_rows, per_shard)):
        pq.write_table(t.slice(s, per_shard), os.path.join(a.out_dir, f"{a.source}-{k:05d}.parquet"),
                       row_group_size=ROWS_PER_RG, compression="snappy")
    d = t.select(["id", "label", "speaker_id", "source", "duration"]).to_pandas()
    pd.DataFrame({"id": d.id, "label": d.label, "source": d.source, "group": d.speaker_id, "fold": "X"}).to_csv(
        os.path.join(a.out_dir, f"folds_{a.source}.csv"), index=False)
    pd.DataFrame([{"dataset_id": a.name, "config_split": "all", "revision": "mdc", "license": a.license, "label": a.label,
                   "url": a.url, "clips": len(d), "speakers": d.speaker_id.nunique(), "hours": round(d.duration.sum() / 3600, 2)}]).to_csv(
        os.path.join(a.out_dir, "manifest.csv"), index=False)
    print(f"kept {len(d)} clips, {d.speaker_id.nunique()} speakers, {d.duration.sum() / 3600:.2f} h")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/voice_overlap.py
"""Do any speakers in a new dataset sound like speakers we already have? (WavLM x-vectors, CPU)

Each speaker gets a centroid from up to --per-speaker clips (first 6 s of each). The threshold is
calibrated on the reference speakers: two disjoint halves of one speaker's clips vs two different
speakers. A new speaker is flagged when its best match beats the 99.5th percentile of the
different-speaker similarities.

    python src/voice_overlap.py --new <dir> --ref <dir> [<dir> ...] --labels ibb,anw,efi --out overlap.csv
"""
import argparse
import glob
import io

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from transformers import AutoFeatureExtractor, WavLMForXVector

SR, SECONDS = 16000, 6


def load(dirs, labels, per_speaker, seed=0):
    rows = []
    for d in dirs:
        for shard in sorted(glob.glob(f"{d}/**/*.parquet", recursive=True)):
            t = pq.read_table(shard, columns=["id", "label", "speaker_id", "source"]).to_pandas()
            t = t[t.label.isin(labels)]
            if len(t):
                t["shard"] = shard
                rows.append(t)
    df = pd.concat(rows).sample(frac=1, random_state=seed)
    df = df[df.groupby("speaker_id").cumcount() < per_speaker]
    waves = {}
    for shard, g in df.groupby("shard"):
        t = pq.read_table(shard, columns=["id", "audio"])
        want = set(g.id)
        for i, cid in enumerate(t.column("id").to_pylist()):
            if cid in want:
                waves[cid] = sf.read(io.BytesIO(t.column("audio")[i].as_py()["bytes"]), dtype="float32")[0][: SECONDS * SR]
    df["wave"] = df.id.map(waves)
    return df.reset_index(drop=True)


def embed(df, fe, model, batch=16):
    out = np.zeros((len(df), 512), np.float32)
    order = np.argsort([len(w) for w in df.wave])
    for b in range(0, len(order), batch):
        idx = order[b:b + batch]
        x = fe([df.wave[i] for i in idx], sampling_rate=SR, return_tensors="pt", padding=True, return_attention_mask=True)
        with torch.no_grad():
            out[idx] = torch.nn.functional.normalize(model(**x).embeddings, dim=-1).numpy()
    return out


def centroid(e):
    c = e.mean(0)
    return c / np.linalg.norm(c)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--new", nargs="+", required=True)
    ap.add_argument("--ref", nargs="+", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--per-speaker", type=int, default=8)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    labels = a.labels.split(",")
    torch.set_num_threads(4)
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
    model = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").eval()

    ref = load(a.ref, labels, 2 * a.per_speaker)
    new = load(a.new, labels, a.per_speaker)
    print(f"ref: {ref.speaker_id.nunique()} speakers / {len(ref)} clips; new: {new.speaker_id.nunique()} speakers / {len(new)} clips", flush=True)
    er, en = embed(ref, fe, model), embed(new, fe, model)

    halves, ref_c = [], {}
    for s, g in ref.groupby("speaker_id"):
        idx = g.index.to_numpy()
        ref_c[s] = (centroid(er[idx]), g.label.iloc[0], g.source.iloc[0])
        if len(idx) >= 4:
            h = len(idx) // 2
            halves.append(float(centroid(er[idx[:h]]) @ centroid(er[idx[h:]])))
    names = list(ref_c)
    C = np.stack([ref_c[s][0] for s in names])
    diff = (C @ C.T)[np.triu_indices(len(names), 1)]
    thr = float(np.percentile(diff, 99.5))
    print(f"same speaker (split halves): median {np.median(halves):.3f}, min {np.min(halves):.3f}; "
          f"different speakers: median {np.median(diff):.3f}, 99.5% {thr:.3f}", flush=True)

    res = []
    for s, g in new.groupby("speaker_id"):
        c = centroid(en[g.index.to_numpy()])
        sims = C @ c
        j = int(np.argmax(sims))
        res.append(dict(new_speaker=s, label=g.label.iloc[0], clips=len(g), best_ref=names[j], best_ref_label=ref_c[names[j]][1],
                        best_ref_source=ref_c[names[j]][2], similarity=round(float(sims[j]), 3), flagged=bool(sims[j] > thr)))
    res = pd.DataFrame(res).sort_values("similarity", ascending=False)
    res.to_csv(a.out, index=False)
    print(res.head(15).to_string(index=False))
    print(f"flagged {int(res.flagged.sum())} of {len(res)} new speakers (threshold {thr:.3f})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_kernel.py
"""Training script for the ladder rungs, run as a Kaggle GPU script kernel.

The run is set by CONFIG below; scripts/launch fills it in per run:
- mode "cv":   train one model per fold in CONFIG["folds"], save out-of-fold logits on
               full clips for the held-out fold (used for CV scores and bias tuning).
- mode "full": train on every clip, save the fp16 model, run the organisers' checker,
               then reload offline and measure latency and memory.

Outputs in /kaggle/working: metrics.json, oof_<fold>.npz (cv), model/ and bench (full).
"""
CONFIG = {"mode": "cv", "folds": ["A", "B", "C"], "folds_file": "folds_v1.csv", "n_layers": 4,
          "steps": 4000, "batch": 32, "crop_min_s": 3.0, "crop_max_s": 6.0, "lr_body": 5e-5, "lr_head": 5e-4,
          "loss": "ce", "tau": 0.0, "group_alpha": 0.5, "aug": False, "ext_cap": 0.5, "eval_per_class": 200, "eval_at": [1.0], "seed": 0}

import glob
import io
import json
import os
import random
import subprocess
import sys
import time
from multiprocessing import Pool

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "soundfile"], check=True)

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

LABELS = ["yor", "ibo", "hau", "ego", "anw", "efi", "ibb", "idu", "igl", "nup", "tiv", "urh", "other"]
L2I = {l: i for i, l in enumerate(LABELS)}
BASE, SR = "ajesujoba/AfriHuBERT", 16000
FOLDS_PATH = glob.glob(f"/kaggle/input/**/{CONFIG['folds_file']}", recursive=True)[0]
INPUT = os.path.dirname(glob.glob("/kaggle/input/**/score.py", recursive=True)[0])  # main data dataset
OUT = "/kaggle/working"
sys.path.insert(0, INPUT)
from score import efficiency, final_score, weighted_f1  # noqa: E402

dev = "cuda"
metrics = {"config": CONFIG, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}


def log(**kw):
    metrics.update(kw)
    print(json.dumps(kw), flush=True)
    with open(f"{OUT}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=1)


# ---------- audio, decoded once into one int16 array ----------
def decode_shard(path):
    """Decode one shard into a single int16 array plus per-clip (id, start, length)."""
    t = pq.read_table(path, columns=["id", "audio"])
    ids, waves = t.column("id").to_pylist(), []
    for a in t.column("audio").to_pylist():
        x, sr = sf.read(io.BytesIO(a["bytes"]), dtype="int16")
        assert sr == SR and x.ndim == 1
        waves.append(x)
    lengths = np.array([len(w) for w in waves])
    starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    return ids, starts, lengths, np.concatenate(waves)


t0 = time.time()
folds = pd.read_csv(FOLDS_PATH)
# Kaggle's mounted input can be very slow for pyarrow's many small reads, so copy the shards
# to local disk with large sequential reads first.
import shutil  # noqa: E402
from concurrent.futures import ThreadPoolExecutor  # noqa: E402

LOCAL = "/tmp/shards"
os.makedirs(LOCAL, exist_ok=True)
shard_paths = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))  # competition + external shards
with ThreadPoolExecutor(6) as ex:
    local_paths = list(ex.map(lambda p: shutil.copyfile(p, f"{LOCAL}/{os.path.basename(p)}"), shard_paths))
log(copy_s=round(time.time() - t0, 1), n_shards=len(local_paths))
SHARDS, where = [], {}  # one int16 array per shard; id -> (shard, start, length)
with Pool(4) as pool:
    for ids, starts, lengths, audio in pool.imap(decode_shard, local_paths):
        for i, s, n in zip(ids, starts, lengths):
            where[i] = (len(SHARDS), int(s), int(n))
        SHARDS.append(audio)
shutil.rmtree(LOCAL, ignore_errors=True)  # local copies are no longer needed once decoded
folds = folds[folds.id.isin(where)].reset_index(drop=True)
# Sampling groups: real speakers for hypa/omnilingual; afrispeech clips count individually,
# so sampling is identical whichever pseudo-speaker clustering the folds use.
folds["sgroup"] = np.where(folds.source == "afrispeech_ipa", folds.id, folds.group)
loc = np.array([where[i] for i in folds.id])  # shard, start, length per row of folds
lens = loc[:, 2]
Y = np.array([L2I[l] for l in folds.label])
EVAL_ONLY = folds.fold.values == "V"  # evaluation-only external clips: never used for training
log(n_clips=len(folds), audio_hours=round(lens.sum() / SR / 3600, 2), decode_s=round(time.time() - t0, 1))


def wave(k, start=0, n=None):
    sh, s, length = loc[k]
    n = length - start if n is None else n
    return SHARDS[sh][s + start:s + start + n].astype(np.float32) / 32768.0


# ---------- sampling: class uniform -> source ~ hours^0.5 -> speaker group ~ size^alpha -> clip ----------
# The source level keeps the few hypa-channel clips of the big languages from being drowned
# out by thousands of afrispeech clips (the test set mixes all three sources).
def sampling_weights(idx):
    d = folds.iloc[idx].assign(n=lens[idx])
    w = np.zeros(len(idx))
    pos = {k: j for j, k in enumerate(d.index)}
    for lab, dl in d.groupby("label"):
        src_w = dl.groupby("source").n.sum() ** 0.5
        src_w = src_w / src_w.sum()
        # External sources together get at most ext_cap of a class, so clean studio corpora
        # (e.g. the 2-speaker Efik set) cannot become the class's dominant "sound".
        ext = src_w.index.str.startswith("ext_")
        if ext.any() and (~ext).any() and src_w[ext].sum() > CONFIG["ext_cap"]:
            src_w[ext] *= CONFIG["ext_cap"] / src_w[ext].sum()
            src_w[~ext] *= (1 - CONFIG["ext_cap"]) / src_w[~ext].sum()
        for src, ds in dl.groupby("source"):
            gsize = ds.groupby("sgroup").size()
            gw = gsize ** CONFIG["group_alpha"]
            gw = gw / gw.sum()
            share = (1 / d.label.nunique()) * src_w[src] * gw[ds.sgroup].values / gsize[ds.sgroup].values
            w[[pos[k] for k in ds.index]] = share
    return w / w.sum()


# ---------- augmentation (rung 3): applied to every class alike ----------
# Erases the per-corpus fingerprints (digital silence, noise floor, spectrum) and adds the test
# conditions the training data lacks (phone band and codec, noise, reverb).
from scipy.signal import butter, fftconvolve, resample_poly, sosfilt  # noqa: E402

PHONE_SOS = butter(4, [300, 3400], btype="bandpass", fs=SR, output="sos")
SPEEDS = [(10, 9), (20, 19), (20, 21), (10, 11)]  # resample ratios for speed 0.9, 0.95, 1.05, 1.1
if CONFIG.get("speed_set") == "wide":  # 0.85-1.15: formant shifts closer to a different speaker's
    SPEEDS += [(20, 17), (20, 23)]
SPEED_P = CONFIG.get("speed_p", 0.5)


def fit_length(x, n):
    if len(x) >= n:
        s = np.random.randint(0, len(x) - n + 1)
        return x[s:s + n]
    return np.pad(x, (0, n - len(x)), mode="wrap")


def rms(x):
    return np.sqrt(np.mean(x ** 2) + 1e-12)


def colored_noise(n):
    white = np.random.randn(n)
    kind = np.random.randint(3)
    if kind == 0:
        return white
    spec = np.fft.rfft(white)
    f = np.arange(len(spec)) + 1.0
    spec /= f ** (0.5 if kind == 1 else 1.0)  # pink or brown
    return np.fft.irfft(spec, n)


def augment(x, babble=None):
    n = len(x)
    x = x.astype(np.float64)
    if np.random.rand() < SPEED_P:  # speed perturbation (tempo and pitch together)
        up, down = SPEEDS[np.random.randint(len(SPEEDS))]
        x = fit_length(resample_poly(x, up, down), n)
    if np.random.rand() < 0.5:  # random smooth EQ, +-6 dB over 8 bands
        spec = np.fft.rfft(x)
        curve = np.interp(np.linspace(0, 7, len(spec)), np.arange(8), np.random.uniform(-6, 6, 8))
        x = np.fft.irfft(spec * 10 ** (curve / 20), n)
    if np.random.rand() < 0.3:  # synthetic room reverb
        rt60 = np.random.uniform(0.2, 0.8)
        t = np.arange(int(rt60 * SR)) / SR
        rir = np.random.randn(len(t)) * np.exp(-6.9 * t / rt60)
        rir[0] = 1.0
        x = fftconvolve(x, rir / np.sqrt(np.sum(rir ** 2)))[:n]
    if np.random.rand() < 0.5:  # background noise or babble
        if babble is not None and np.random.rand() < 0.3:
            noise, snr = fit_length(babble.astype(np.float64), n), np.random.uniform(10, 25)
        else:
            noise, snr = colored_noise(n), np.random.uniform(0, 20)
        x = x + noise * rms(x) / rms(noise) / 10 ** (snr / 20)
    if np.random.rand() < 0.25:  # phone channel: 300-3400 Hz, 8 kHz, mu-law 8-bit
        x = sosfilt(PHONE_SOS, x)
        x = resample_poly(resample_poly(x, 1, 2), 2, 1)[:n]
        peak = np.max(np.abs(x)) + 1e-9
        mu = np.sign(x / peak) * np.log1p(255 * np.abs(x / peak)) / np.log(256)
        mu = np.round(mu * 127) / 127
        x = peak * np.sign(mu) * ((256 ** np.abs(mu)) - 1) / 255
    # always: a random noise floor, so exact digital zeros never survive
    x = x + np.random.randn(n) * 10 ** (np.random.uniform(-75, -45) / 20)
    return x.astype(np.float32)


def batch_stream(tasks, pool, window=48):
    """Yield batches in order, preparing at most two windows ahead. An unbounded imap lets the
    workers outrun a slow GPU step and fill RAM with finished batches (D3 was OOM-killed)."""
    if pool is None:
        yield from map(make_batch, tasks)
        return
    pending = pool.map_async(make_batch, tasks[:window])
    for s in range(0, len(tasks), window):
        current = pending.get()
        if s + window < len(tasks):
            pending = pool.map_async(make_batch, tasks[s + window:s + 2 * window])
        yield from current


def make_batch(task):
    """One training batch: (clip indices, start offsets, crop length(s), babble indices, seed)."""
    ks, starts, n, babble_ks, seed = task
    np.random.seed(seed)
    ns = n if isinstance(n, list) else [n] * len(ks)
    out = []
    for k, s, n, bk in zip(ks, starts, ns, babble_ks):
        x = wave(k, s, n)
        out.append(augment(x, wave(bk)) if CONFIG["aug"] else x)
    return out


# ---------- test-like conditions for local evaluation (deterministic per clip) ----------
def degrade(x, cond, seed):
    """'noisy': coloured noise at 0-10 dB SNR, half the time with reverb. 'phone': 300-3400 Hz,
    8 kHz, mu-law 8-bit, low noise floor. Mirrors the leaderboard's clean/noisy/phone split."""
    if cond == "clean":
        return x
    rs = np.random.get_state()
    np.random.seed(seed)
    y = x.astype(np.float64)
    n = len(y)
    if cond == "noisy":
        if np.random.rand() < 0.5:
            rt60 = np.random.uniform(0.3, 0.9)
            t = np.arange(int(rt60 * SR)) / SR
            rir = np.random.randn(len(t)) * np.exp(-6.9 * t / rt60)
            rir[0] = 1.0
            y = fftconvolve(y, rir / np.sqrt(np.sum(rir ** 2)))[:n]
        noise = colored_noise(n)
        y = y + noise * rms(y) / rms(noise) / 10 ** (np.random.uniform(0, 10) / 20)
    else:
        y = sosfilt(PHONE_SOS, y)
        y = resample_poly(resample_poly(y, 1, 2), 2, 1)[:n]
        peak = np.max(np.abs(y)) + 1e-9
        mu = np.round(np.sign(y / peak) * np.log1p(255 * np.abs(y / peak)) / np.log(256) * 127) / 127
        y = peak * np.sign(mu) * ((256 ** np.abs(mu)) - 1) / 255 + np.random.randn(n) * 10 ** (-60 / 20)
    np.random.set_state(rs)
    return y.astype(np.float32)


# ---------- model ----------
# backbone "hubert" (AfriHuBERT, raw-audio front end) or "whisper" (a Whisper encoder on log-mel
# features with its window cut to window_s seconds: much cheaper on CPU, and the feature
# extractor itself truncates longer clips to the window, so inference cost is bounded).
from transformers import AutoFeatureExtractor, HubertForSequenceClassification  # noqa: E402
from transformers import WhisperFeatureExtractor, WhisperForAudioClassification  # noqa: E402

WHISPER = CONFIG.get("backbone") == "whisper"
if WHISPER:
    fe = WhisperFeatureExtractor.from_pretrained(CONFIG["whisper_name"], chunk_length=CONFIG["window_s"])
else:
    fe = AutoFeatureExtractor.from_pretrained(BASE)


def inputs(waves):
    if WHISPER:
        return fe(waves, sampling_rate=SR, return_tensors="pt", device=dev).input_features.to(dev)
    return fe(waves, sampling_rate=SR, return_tensors="pt", padding=True).input_values.to(dev)


def make_model():
    kw = dict(num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I, use_weighted_layer_sum=True)
    if not WHISPER:
        model = HubertForSequenceClassification.from_pretrained(
            BASE, num_hidden_layers=CONFIG["n_layers"], **kw,
            mask_time_prob=CONFIG.get("mask_time_prob", 0.05), mask_feature_prob=CONFIG.get("mask_feature_prob", 0.0))
        model.freeze_feature_encoder()
        return model
    model = WhisperForAudioClassification.from_pretrained(CONFIG["whisper_name"], **kw)
    enc, frames = model.encoder, int(CONFIG["window_s"] * 50)  # encoder runs at 50 positions per second
    old = enc.embed_positions
    new = torch.nn.Embedding(frames, old.embedding_dim)
    new.weight.data.copy_(old.weight.data[:frames])
    new.weight.requires_grad_(old.weight.requires_grad)
    enc.embed_positions = new
    enc.max_source_positions = model.config.max_source_positions = frames
    return model


def train(train_idx, seed, on_eval=None, init_seed=None):
    """init_seed (default: seed) sets the new head's initialisation; seed sets data order, crops and
    augmentation. Models that share init_seed can be weight-averaged (same basin, aligned heads)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed if init_seed is None else init_seed)
    model = make_model().to(dev)
    if init_seed is not None and init_seed != seed:
        torch.manual_seed(seed)
    is_head = lambda n: n.startswith(("projector", "classifier", "layer_weights"))
    params = [{"params": [p for n, p in model.named_parameters() if p.requires_grad and not is_head(n)], "lr": CONFIG["lr_body"]},
              {"params": [p for n, p in model.named_parameters() if is_head(n)], "lr": CONFIG["lr_head"]}]
    opt = torch.optim.AdamW(params, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[CONFIG["lr_body"], CONFIG["lr_head"]],
                                                total_steps=CONFIG["steps"], pct_start=0.08)
    scaler = torch.cuda.amp.GradScaler()
    w = sampling_weights(train_idx)
    draws = np.random.choice(train_idx, size=CONFIG["steps"] * CONFIG["batch"], p=w)
    # Logit adjustment: add tau * log(prior of the sampled batch distribution) during training.
    prior = np.bincount(Y[draws], minlength=len(LABELS)) / len(draws)
    log_prior = torch.tensor(np.log(np.maximum(prior, 1e-8)), dtype=torch.float32, device=dev)
    eval_steps = {max(1, int(round(f * CONFIG["steps"]))) for f in CONFIG["eval_at"]}
    model.train()
    t0 = time.time()
    tasks = []
    for step in range(CONFIG["steps"]):
        b = draws[step * CONFIG["batch"]:(step + 1) * CONFIG["batch"]]
        if WHISPER:  # one crop length per clip: the feature extractor pads each to the window
            n = [int(min(random.uniform(CONFIG["crop_min_s"], CONFIG["crop_max_s"]) * SR, lens[k])) for k in b]
            starts = [random.randint(0, lens[k] - nk) for k, nk in zip(b, n)]
        else:
            n = int(min(random.uniform(CONFIG["crop_min_s"], CONFIG["crop_max_s"]) * SR, lens[b].min()))
            starts = [random.randint(0, lens[k] - n) for k in b]
        tasks.append((b, starts, n, np.random.choice(train_idx, len(b)), seed * 100003 + step))
    pool = Pool(3) if CONFIG["aug"] else None
    batches = batch_stream(tasks, pool)
    for step, (task, batch) in enumerate(zip(tasks, batches)):
        b = task[0]
        y = torch.tensor(Y[b], device=dev)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(inputs(batch)).logits.float()
        if CONFIG["loss"] == "logit_adjusted":
            logits = logits + CONFIG["tau"] * log_prior
        loss = torch.nn.functional.cross_entropy(logits, y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if (step + 1) % 250 == 0:
            print(f"step {step + 1}/{CONFIG['steps']} loss {loss.item():.3f} {time.time() - t0:.0f}s", flush=True)
        if on_eval and (step + 1) in eval_steps:
            on_eval(model, step + 1)
            model.train()
    if pool:
        pool.close()
    return model, round(time.time() - t0, 1)


@torch.no_grad()
def full_clip_logits(model, idx):
    model.eval()
    out = np.zeros((len(idx), len(LABELS)), dtype=np.float32)
    for j, k in enumerate(idx):
        with torch.autocast("cuda", dtype=torch.float16):
            out[j] = model(inputs([wave(k)])).logits.float().cpu().numpy()[0]
    return out


@torch.no_grad()
def condition_logits(model, idx):
    model.eval()
    out = {c: np.zeros((len(idx), len(LABELS)), dtype=np.float32) for c in ("clean", "noisy", "phone")}
    for j, k in enumerate(idx):
        x = wave(k)
        for ci, c in enumerate(out):
            with torch.autocast("cuda", dtype=torch.float16):
                out[c][j] = model(inputs([degrade(x, c, 7919 * int(k) + ci)])).logits.float().cpu().numpy()[0]
    return out


def cv_split(f, rng):
    tr = np.where((folds.fold.values != f) & ~EVAL_ONLY)[0]
    ho_all = np.where(folds.fold.values == f)[0]
    # evaluation subset: up to eval_per_class held-out clips per class (competition clips only)
    ho = np.concatenate([rng.permutation(ho_all[Y[ho_all] == c])[:CONFIG["eval_per_class"]] for c in range(len(LABELS))]).astype(int)
    va_all = np.where(EVAL_ONLY)[0]
    va = np.concatenate([rng.permutation(va_all[Y[va_all] == c])[:CONFIG["eval_per_class"] // 2] for c in range(len(LABELS))]).astype(int)
    return tr, ho, va


def evaluate(model, f, tag, ho, va):
    t1 = time.time()
    lg = condition_logits(model, ho)
    np.savez_compressed(f"{OUT}/oof_{f}_{tag}.npz", ids=folds.id.values[ho], **lg)
    if len(va):
        lv = condition_logits(model, va)
        np.savez_compressed(f"{OUT}/va_{f}_{tag}.npz", ids=folds.id.values[va], **lv)
    res = {}
    for c, l in lg.items():
        wf1, _ = weighted_f1(folds.label.values[ho].tolist(), [LABELS[i] for i in l.argmax(1)])
        res[c] = {"weighted_f1": round(wf1, 4), "acc": round(float((l.argmax(1) == Y[ho]).mean()), 4)}
    log(**{f"fold_{f}_{tag}": {**res, "n_eval": len(ho), "eval_s": round(time.time() - t1, 1)}})


if CONFIG["mode"] == "cv":
    rng = np.random.default_rng(CONFIG["seed"])
    for f in CONFIG["folds"]:
        tr, ho, va = cv_split(f, rng)
        model, train_s = train(tr, CONFIG["seed"], on_eval=lambda m, step, f=f, ho=ho, va=va: evaluate(m, f, f"s{step}", ho, va))
        log(**{f"fold_{f}_train_s": train_s})
        del model
        torch.cuda.empty_cache()


def save_full(model, train_s, name="model"):
    log(**{f"{name}_train_s": train_s})
    model_dir = f"{OUT}/{name}"
    model.half().save_pretrained(model_dir, safe_serialization=True)
    fe.save_pretrained(model_dir)
    chk = subprocess.run([sys.executable, f"{INPUT}/check_model_folder.py", model_dir], capture_output=True, text=True)
    log(**{f"{name}_files": sorted(os.listdir(model_dir)), f"{name}_checker": chk.stdout.strip().splitlines(), f"{name}_checker_ok": chk.returncode == 0})
    del model
    torch.cuda.empty_cache()
    # Organisers' side, in a fresh process: offline load with default dtype, per-clip
    # inference on 300 clips of mixed length, warm-up, median of 3 passes, peak memory.
    rng = np.random.default_rng(0)
    bench_idx = rng.choice(len(folds), 300, replace=False)
    np.save(f"{OUT}/bench_audio.npy", np.concatenate([wave(k) for k in bench_idx]).astype(np.float32))
    np.save(f"{OUT}/bench_lens.npy", lens[bench_idx])
    bench = r'''
import json, os, resource, statistics, sys, time
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
import numpy as np, torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
model_dir, out = sys.argv[1:3]
audio, lens = np.load(f"{out}/bench_audio.npy"), np.load(f"{out}/bench_lens.npy")
offs = np.concatenate([[0], np.cumsum(lens)])
waves = [audio[offs[i]:offs[i + 1]] for i in range(len(lens))]
fe = AutoFeatureExtractor.from_pretrained(model_dir)
m = AutoModelForAudioClassification.from_pretrained(model_dir).to("cuda").eval()
torch.cuda.reset_peak_memory_stats()
def x_of(w):
    f = fe(w, sampling_rate=16000, return_tensors="pt")
    return f["input_features"] if "input_features" in f else f["input_values"]
def run():
    t = time.time()
    with torch.no_grad():
        for w in waves:
            m(x_of(w).to("cuda", m.dtype)).logits.argmax(-1).item()
    torch.cuda.synchronize()
    return (time.time() - t) / len(waves)
run()
lat = statistics.median(run() for _ in range(3))
# CPU-only runner: per-clip time on 1 and 4 threads, and the projected time for 6,659 test clips
mc = AutoModelForAudioClassification.from_pretrained(model_dir).eval()
cpu = {}
for th in (1, 4):
    torch.set_num_threads(th)
    with torch.no_grad():
        mc(x_of(waves[0]))
        t = time.time()
        for w in waves[:100]:
            mc(x_of(w)).logits.argmax(-1).item()
    per = (time.time() - t) / 100
    cpu[f"cpu_{th}t_s_per_clip"] = round(per, 4)
    cpu[f"cpu_{th}t_6659_clips_s"] = round(per * 6659)
print("BENCH " + json.dumps({"loaded_dtype": str(m.dtype), "latency_s": round(lat, 5), **cpu,
      "gpu_peak_gb": round(torch.cuda.max_memory_reserved() / 1e9, 3),
      "ram_peak_gb": round(int([l for l in open("/proc/self/status") if l.startswith("VmHWM")][0].split()[1]) / 1e6, 3),
      "params_m": round(sum(p.numel() for p in m.parameters()) / 1e6, 2)}))
'''
    with open(f"{OUT}/bench.py", "w") as f:
        f.write(bench)
    r = subprocess.run([sys.executable, f"{OUT}/bench.py", model_dir, OUT], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("BENCH ")]
    if line:
        b = json.loads(line[0][6:])
        repo_gb = sum(os.path.getsize(f"{model_dir}/{n}") for n in os.listdir(model_dir)) / 1e9
        eff, parts = efficiency(b["params_m"], repo_gb, max(b["gpu_peak_gb"], b["ram_peak_gb"]), b["latency_s"])
        b.update(repo_gb=round(repo_gb, 4), efficiency=round(eff, 3), eff_parts={k: round(v, 3) for k, v in parts.items()})
        log(**{f"{name}_bench": b})
    else:
        log(**{f"{name}_bench_error": r.stderr[-2000:]})
    for fn in ["bench_audio.npy", "bench_lens.npy"]:
        os.remove(f"{OUT}/{fn}")


def average_weights(sds):
    return {k: (sum(sd[k] for sd in sds) / len(sds)) if sds[0][k].is_floating_point() else sds[0][k] for k in sds[0]}


if CONFIG["mode"] == "full":
    model, train_s = train(np.where(~EVAL_ONLY)[0], CONFIG["seed"], init_seed=CONFIG.get("init_seed"))
    save_full(model, train_s)
elif CONFIG["mode"] == "full_multi":  # several whisper variants in one run: [[name, window_s, lr_body, out_dir], ...]
    for wname, win, lr, out_name in CONFIG["variants"]:
        CONFIG.update(whisper_name=wname, window_s=win, lr_body=lr, crop_max_s=win)
        fe = WhisperFeatureExtractor.from_pretrained(wname, chunk_length=win)
        model, train_s = train(np.where(~EVAL_ONLY)[0], CONFIG["seed"])
        save_full(model, train_s, out_name)
        del model
        torch.cuda.empty_cache()
elif CONFIG["mode"] == "cv_soup":
    # Weight averaging check: per fold, train one model per seed in soup_seeds (shared head
    # initialisation), score each and their weight average on the held-out fold. Optionally
    # then train the full model for one more seed, to be averaged with an existing full model.
    rng = np.random.default_rng(CONFIG["seed"])
    init = CONFIG.get("init_seed", 0)
    for f in CONFIG["folds"]:
        tr, ho, va = cv_split(f, rng)
        sds = []
        for s in CONFIG["soup_seeds"]:
            model, train_s = train(tr, s, init_seed=init)
            log(**{f"fold_{f}_seed{s}_train_s": train_s})
            evaluate(model, f, f"seed{s}", ho, va)
            sds.append({k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()})
            if len(sds) < len(CONFIG["soup_seeds"]):
                del model
                torch.cuda.empty_cache()
        if len(sds) > 1:
            model.load_state_dict(average_weights(sds))
            evaluate(model, f, "soup", ho, va)
        del model, sds
        torch.cuda.empty_cache()
    if CONFIG.get("full_seed") is not None:  # all rows, V included: the same training set as folds_final
        model, train_s = train(np.arange(len(folds)), CONFIG["full_seed"], init_seed=init)
        save_full(model, train_s)

log(done=True)


In [ ]:
%%writefile src/launch_kernel.py
"""Push train_kernel.py to Kaggle as a private GPU script kernel with a given CONFIG.

    KAGGLE_API_TOKEN=... python src/launch_kernel.py <kaggle-user> <slug> '<json overrides>'

The CONFIG line in train_kernel.py is replaced with the defaults updated by the overrides,
so each run's exact settings are stored in its own kernel version.
"""
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

user, slug, overrides = sys.argv[1], sys.argv[2], json.loads(sys.argv[3])
datasets = overrides.pop("datasets", ["dsn2026-lid-data", "dsn2026-lid-meta"])
src = (Path(__file__).parent / "train_kernel.py").read_text()
m = re.search(r"^CONFIG = (\{.*?\})\n", src, re.S | re.M)
config = eval(m.group(1))  # the literal dict at the top of our own script
config.update(overrides)
src = src[:m.start(1)] + json.dumps(config).replace("false", "False").replace("true", "True") + src[m.end(1):]

with tempfile.TemporaryDirectory() as d:
    (Path(d) / "train_kernel.py").write_text(src)
    (Path(d) / "kernel-metadata.json").write_text(json.dumps({
        "id": f"{user}/{slug}", "title": slug, "code_file": "train_kernel.py", "language": "python",
        "kernel_type": "script", "is_private": True, "enable_gpu": True, "enable_internet": True,
        "dataset_sources": [f"{user}/{d}" for d in datasets], "competition_sources": [], "kernel_sources": []}))
    kaggle = str(Path(sys.executable).parent / "kaggle")
    r = subprocess.run([kaggle, "kernels", "push", "-p", d, "--accelerator", "NvidiaTeslaT4"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    print("config:", json.dumps(config))


In [ ]:
%%writefile src/predict_lb.py
"""Local leaderboard: predict a recipe's public-leaderboard score from its CV predictions.

Inputs are the out-of-fold files from a CV run (oof_<fold>_s<step>.npz with clean, noisy and
phone logits for speaker-held-out clips). The predicted score imitates the public test:
- class mix: the clip counts of the public test split, as reported for D1 (2,358 clips)
- conditions: clean / noisy / phone shares fitted to D1's leaderboard accuracies
- efficiency: the official formula with D1's measured memory and latency for HuBERT models
Bootstrap resampling of clips gives a 90% interval. nup has no held-out speaker (one speaker
in all training data), so its F1 cannot be estimated and is reported as missing; its false
positives on other classes are still counted.

    python src/predict_lb.py <folds.csv> oof_A_s4000.npz oof_B_s4000.npz ... [--layers 4] [--offsets f.json]
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
from score import LABELS, WEIGHTS, efficiency, final_score  # noqa: E402

TEST_COUNTS = {"yor": 317, "ibo": 217, "hau": 295, "ego": 155, "anw": 123, "efi": 45, "ibb": 23, "idu": 171,
               "igl": 55, "nup": 52, "tiv": 242, "urh": 129, "other": 534}
CONDITIONS = {"clean": 0.47, "noisy": 0.265, "phone": 0.265}  # fitted to D1: 0.641 / 0.210 / 0.436 -> 0.4716 overall
PARAMS = {4: (37.9, 0.076), 6: (51.8, 0.104), 8: (66.0, 0.132), 12: (94.4, 0.189)}  # params M, repo GB (fp16)
MEM_GB, LATENCY_S = 1.72, 0.004  # D1 on the organisers' hardware (HuBERT family)

W = np.array([WEIGHTS[c] for c in LABELS])


def f1_per_class(y, p):
    k = len(LABELS)
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    return np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0), cm


def tune_offsets(L, y, present, draws=40, sweeps=4, span=3.0, seed=1):
    """Coordinate ascent on per-class logit offsets, maximising the mean weighted F1 over fixed
    bootstrap draws of a test-like set (public-test class counts and condition mix)."""
    rng = np.random.default_rng(seed)
    conds = list(CONDITIONS)
    probs = np.array([CONDITIONS[c] for c in conds])
    sets = []
    for _ in range(draws):
        lg, yy = [], []
        for i in np.where(present)[0]:
            idx = np.where(y == i)[0]
            n = TEST_COUNTS[LABELS[i]]
            pick, cond = rng.choice(idx, n, replace=True), rng.choice(len(conds), n, p=probs)
            lg.append(np.stack([L[conds[c]][k] for k, c in zip(pick, cond)]))
            yy.append(np.full(n, i))
        sets.append((np.concatenate(lg), np.concatenate(yy)))

    def objective(off):
        tot = 0.0
        for lg, yy in sets:
            f1, _ = f1_per_class(yy, (lg + off).argmax(1))
            tot += (W[present] * f1[present]).sum() / W[present].sum()
        return tot / len(sets)

    off = np.zeros(len(LABELS))
    best = objective(off)
    print(f"tuning: mean weighted F1 before {best:.4f}")
    grid = np.linspace(-span, span, 25)
    for sweep in range(sweeps):
        improved = False
        for c in range(len(LABELS)):
            cand = off.copy()
            scores = []
            for g in grid:
                cand[c] = off[c] + g * 0.5 ** sweep
                scores.append(objective(cand))
            j = int(np.argmax(scores))
            if scores[j] > best + 1e-5:
                off[c] += grid[j] * 0.5 ** sweep
                best, improved = scores[j], True
        print(f"  sweep {sweep}: {best:.4f}")
        if not improved:
            break
    return off - off.mean()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folds")
    ap.add_argument("oof", nargs="+")
    ap.add_argument("--layers", type=int, default=4)
    ap.add_argument("--offsets")
    ap.add_argument("--boot", type=int, default=300)
    ap.add_argument("--json")
    ap.add_argument("--va", nargs="*", default=[], help="evaluation-only VoiceAfrica files (va_<fold>_s<step>.npz)")
    ap.add_argument("--va-classes", default="nup", help="classes whose pool is filled from --va (default: nup only)")
    ap.add_argument("--tune-out", help="tune per-class offsets for the test mix and write them here (JSON)")
    a = ap.parse_args()

    labels = pd.read_csv(a.folds).set_index("id")["label"]
    ids, L = [], {c: [] for c in CONDITIONS}
    for f in a.oof:
        z = np.load(f, allow_pickle=True)
        ids.extend(z["ids"].tolist())
        for c in CONDITIONS:
            L[c].append(z[c])
    va_classes = set(a.va_classes.split(","))
    for f in a.va:  # add held-out VoiceAfrica speakers for the chosen classes (e.g. nup, which has no CV speaker)
        z = np.load(f, allow_pickle=True)
        keep = np.array([labels.loc[i] in va_classes for i in z["ids"]])
        ids.extend(z["ids"][keep].tolist())
        for c in CONDITIONS:
            L[c].append(z[c][keep])
    L = {c: np.concatenate(v) for c, v in L.items()}
    y = np.array([LABELS.index(l) for l in labels.loc[ids]])
    off = np.zeros(len(LABELS))
    if a.offsets:
        o = json.loads(Path(a.offsets).read_text())["offsets"]
        off = np.array([o[c] for c in LABELS])
    present = np.array([np.any(y == i) for i in range(len(LABELS))])
    if a.tune_out:
        off = tune_offsets(L, y, present)
        Path(a.tune_out).write_text(json.dumps({"offsets": dict(zip(LABELS, off.round(4).tolist())),
                                                "note": "tuned on CV out-of-fold logits under the public-test class and condition mix"}, indent=1))
        print("tuned offsets:", json.dumps(dict(zip(LABELS, off.round(2).tolist()))))
    pred = {c: (L[c] + off).argmax(1) for c in CONDITIONS}

    rng = np.random.default_rng(0)
    conds = list(CONDITIONS)
    probs = np.array([CONDITIONS[c] for c in conds])
    by_class = {i: np.where(y == i)[0] for i in range(len(LABELS)) if present[i]}
    wf1s, f1s, accs = [], [], {c: [] for c in conds}
    for _ in range(a.boot):
        yy, pp, cc = [], [], []
        for i, idx in by_class.items():
            n = TEST_COUNTS[LABELS[i]]
            pick = rng.choice(idx, n, replace=True)
            cond = rng.choice(len(conds), n, p=probs)
            yy.append(np.full(n, i))
            cc.append(cond)
            pp.append(np.array([pred[conds[c]][k] for k, c in zip(pick, cond)]))
        yy, pp, cc = np.concatenate(yy), np.concatenate(pp), np.concatenate(cc)
        f1, _ = f1_per_class(yy, pp)
        wf1s.append((W[present] * f1[present]).sum() / W[present].sum())
        f1s.append(f1)
        for j, c in enumerate(conds):
            accs[c].append(float((yy[cc == j] == pp[cc == j]).mean()))
    wf1s, f1s = np.array(wf1s), np.array(f1s)
    params, repo = PARAMS[a.layers]
    eff, parts = efficiency(params, repo, MEM_GB, LATENCY_S)
    finals = np.array([final_score(w, eff) for w in wf1s])
    lo, hi = np.percentile(finals, [5, 95])

    print(f"OOF clips: {len(y)} (classes without held-out speakers: {[LABELS[i] for i in np.where(~present)[0]]})")
    print(f"predicted weighted F1: {wf1s.mean():.3f}  (90%: {np.percentile(wf1s, 5):.3f}-{np.percentile(wf1s, 95):.3f})")
    print(f"efficiency {eff:.3f} -> predicted FINAL: {finals.mean():.4f}  (90%: {lo:.4f}-{hi:.4f})")
    print("accuracy by condition: " + ", ".join(f"{c} {np.mean(v):.3f}" for c, v in accs.items()))
    print("per-class F1: " + ", ".join(f"{c} {'n/a' if not present[i] else f'{f1s[:, i].mean():.2f}'}" for i, c in enumerate(LABELS)))
    if a.json:
        Path(a.json).write_text(json.dumps({
            "pred_weighted_f1": float(wf1s.mean()), "pred_final": float(finals.mean()), "final_90": [float(lo), float(hi)],
            "efficiency": eff, "acc_by_condition": {c: float(np.mean(v)) for c, v in accs.items()},
            "f1": {c: (None if not present[i] else float(f1s[:, i].mean())) for i, c in enumerate(LABELS)}}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/crossfit_offsets.py
"""Honest check of decision offsets: tune on two CV folds, score on the third.

For each held-out fold, offsets are tuned (predict_lb.tune_offsets) on the other two folds'
out-of-fold logits, then the held-out fold is scored with offsets scaled by each shrink factor.
Offsets for classes listed in --zero are forced to 0 (e.g. nup, whose only held-out speakers are
VoiceAfrica, a different recording domain from the test set).

    python src/crossfit_offsets.py <folds.csv> <oof_dir> <step> [--zero nup] [--va]
"""
import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
from predict_lb import CONDITIONS, TEST_COUNTS, W, f1_per_class, tune_offsets  # noqa: E402
from score import LABELS  # noqa: E402


def load(paths, labels, keep=None):
    ids, L = [], {c: [] for c in CONDITIONS}
    for p in paths:
        z = np.load(p, allow_pickle=True)
        m = np.ones(len(z["ids"]), bool) if keep is None else np.array([labels.loc[i] in keep for i in z["ids"]])
        ids.extend(z["ids"][m].tolist())
        for c in CONDITIONS:
            L[c].append(z[c][m])
    return ids, {c: np.concatenate(v) for c, v in L.items()}


def score(L, y, off, boot=150, seed=0):
    present = np.array([np.any(y == i) for i in range(len(LABELS))])
    rng = np.random.default_rng(seed)
    conds, probs = list(CONDITIONS), np.array(list(CONDITIONS.values()))
    pred = {c: (L[c] + off).argmax(1) for c in conds}
    out = []
    for _ in range(boot):
        yy, pp = [], []
        for i in np.where(present)[0]:
            idx = np.where(y == i)[0]
            n = TEST_COUNTS[LABELS[i]]
            pick, cond = rng.choice(idx, n, replace=True), rng.choice(len(conds), n, p=probs)
            yy.append(np.full(n, i))
            pp.append(np.array([pred[conds[c]][k] for k, c in zip(pick, cond)]))
        f1, _ = f1_per_class(np.concatenate(yy), np.concatenate(pp))
        out.append((W[present] * f1[present]).sum() / W[present].sum())
    return float(np.mean(out))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folds")
    ap.add_argument("oof_dir")
    ap.add_argument("step")
    ap.add_argument("--zero", default="nup")
    ap.add_argument("--va", action="store_true", help="add held-out VoiceAfrica nup clips to every pool")
    a = ap.parse_args()
    labels = pd.read_csv(a.folds).set_index("id")["label"]
    d, folds = Path(a.oof_dir), ["A", "B", "C"]
    zero = [LABELS.index(c) for c in a.zero.split(",") if c]
    shrinks = [0.0, 0.25, 0.5, 0.75, 1.0]
    res = {s: [] for s in shrinks}
    for f in folds:
        others = [x for x in folds if x != f]
        tr_paths = [d / f"oof_{x}_s{a.step}.npz" for x in others]
        te_paths = [d / f"oof_{f}_s{a.step}.npz"]
        ids_tr, Ltr = load(tr_paths, labels)
        ids_te, Lte = load(te_paths, labels)
        if a.va:
            for ids_, L_, xs in ((ids_tr, Ltr, others), (ids_te, Lte, [f])):
                vi, vL = load([d / f"va_{x}_s{a.step}.npz" for x in xs], labels, keep={"nup"})
                ids_.extend(vi)
                for c in CONDITIONS:
                    L_[c] = np.concatenate([L_[c], vL[c]])
        ytr = np.array([LABELS.index(l) for l in labels.loc[ids_tr]])
        yte = np.array([LABELS.index(l) for l in labels.loc[ids_te]])
        present = np.array([np.any(ytr == i) for i in range(len(LABELS))])
        off = tune_offsets(Ltr, ytr, present, draws=20, sweeps=3)
        off[zero] = 0.0
        for s in shrinks:
            res[s].append(score(Lte, yte, s * off))
        print(f"held-out fold {f}: " + ", ".join(f"x{s}: {res[s][-1]:.4f}" for s in shrinks), flush=True)
    print("MEAN over held-out folds: " + ", ".join(f"x{s}: {np.mean(v):.4f}" for s, v in res.items()))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/average_models.py
"""Average the weights of fine-tuned models that share a base model and head initialisation.

The result is one ordinary model (same architecture, size and speed as each input).
Averaging is done in fp32 and saved in the first model's dtype.

    python src/average_models.py <out_dir> <model_dir> <model_dir> [...]
"""
import shutil
import sys
from pathlib import Path

import torch
from safetensors.torch import load_file, save_file


def main():
    out, dirs = Path(sys.argv[1]), [Path(d) for d in sys.argv[2:]]
    sds = [load_file(d / "model.safetensors") for d in dirs]
    assert all(sd.keys() == sds[0].keys() for sd in sds), "different architectures"
    avg = {}
    for k, v in sds[0].items():
        avg[k] = (sum(sd[k].float() for sd in sds) / len(sds)).to(v.dtype) if v.is_floating_point() else v
    out.mkdir(parents=True, exist_ok=True)
    save_file(avg, out / "model.safetensors", metadata={"format": "pt"})
    for name in ("config.json", "preprocessor_config.json"):
        shutil.copy(dirs[0] / name, out / name)
    a, b = (torch.cat([sd[k].double().flatten() for k in sd if sd[k].is_floating_point()]) for sd in sds[:2])
    print(f"averaged {len(sds)} models -> {out}; first two: weight cosine {float(a @ b / (a.norm() * b.norm())):.6f}, "
          f"relative distance {float((a - b).norm() / a.norm()):.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/tune_bias.py
"""Rung 2: tune per-class logit offsets on out-of-fold predictions, then bake them into a model.

Tuning: coordinate ascent over one additive offset per class, maximising the competition's
fixed-share weighted F1 on pooled out-of-fold logits. Each candidate is scored as the mean
over speaker-group bootstrap resamples, so offsets that only fit a few speakers are not chosen.
Classes absent from the out-of-fold set (nup: one speaker, never held out) keep offset 0.

    python src/tune_bias.py tune <folds.csv> <oof_A.npz> [<oof_B.npz> ...] --out offsets.json
    python src/tune_bias.py apply <model_dir> offsets.json <out_model_dir>

Applying adds the offsets to classifier.bias in model.safetensors; the model stays a standard
HubertForSequenceClassification and argmax decisions change exactly as tuned.
"""
import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
from score import LABELS, WEIGHTS  # noqa: E402

W = np.array([WEIGHTS[c] for c in LABELS])


def weighted_f1(y, pred, present):
    k = len(LABELS)
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    return float((W[present] * f1[present]).sum() / W[present].sum())


def tune(args):
    folds = pd.read_csv(args.folds).set_index("id")
    ids, logits = [], []
    for f in args.oof:
        z = np.load(f, allow_pickle=True)
        ids.extend(z["ids"].tolist())
        logits.append(z["logits"])
    logits = np.concatenate(logits)
    y = np.array([LABELS.index(l) for l in folds.loc[ids, "label"]])
    groups = folds.loc[ids, "group"].values
    present = np.isin(np.arange(len(LABELS)), np.unique(y))

    rng = np.random.default_rng(0)
    ug = np.unique(groups)
    boots = []
    for _ in range(args.boot):
        pick = rng.choice(ug, len(ug), replace=True)
        boots.append(np.concatenate([np.where(groups == g)[0] for g in pick]))

    def objective(off):
        pred = (logits + off).argmax(1)
        return np.mean([weighted_f1(y[b], pred[b], present) for b in boots])

    off = np.zeros(len(LABELS))
    base = objective(off)
    best = base
    grid = np.linspace(-args.span, args.span, 25)
    for sweep in range(args.sweeps):
        improved = False
        for c in range(len(LABELS)):
            if not present[c]:
                continue
            cand = off.copy()
            scores = []
            for g in grid:
                cand[c] = off[c] + g * (0.5 ** sweep)
                scores.append(objective(cand))
            j = int(np.argmax(scores))
            if scores[j] > best + 1e-5:
                off[c] += grid[j] * (0.5 ** sweep)
                best = scores[j]
                improved = True
        print(f"sweep {sweep}: bootstrap mean F1 {best:.4f}")
        if not improved:
            break
    off -= off[present].mean()  # only differences matter; keep the vector centred
    plain = weighted_f1(y, logits.argmax(1), present)
    tuned = weighted_f1(y, (logits + off).argmax(1), present)
    print(f"pooled OOF weighted F1: {plain:.4f} -> {tuned:.4f}  (bootstrap mean {base:.4f} -> {best:.4f})")
    res = {"offsets": dict(zip(LABELS, off.round(4).tolist())), "oof_f1_before": plain, "oof_f1_after": tuned,
           "n_clips": len(y), "absent_classes": [LABELS[i] for i in np.where(~present)[0]]}
    Path(args.out).write_text(json.dumps(res, indent=1))
    print(json.dumps(res["offsets"]))


def apply(args):
    import torch
    from safetensors.torch import load_file, save_file
    src, out = Path(args.model_dir), Path(args.out_dir)
    shutil.copytree(src, out, dirs_exist_ok=True)
    off = json.loads(Path(args.offsets).read_text())["offsets"]
    t = load_file(str(src / "model.safetensors"))
    b = t["classifier.bias"]
    t["classifier.bias"] = (b.float() + torch.tensor([off[c] for c in LABELS])).to(b.dtype)
    save_file(t, str(out / "model.safetensors"), metadata={"format": "pt"})
    print("classifier.bias", b.float().numpy().round(3).tolist(), "->", t["classifier.bias"].float().numpy().round(3).tolist())


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("tune")
    t.add_argument("folds")
    t.add_argument("oof", nargs="+")
    t.add_argument("--out", required=True)
    t.add_argument("--boot", type=int, default=30)
    t.add_argument("--span", type=float, default=3.0)
    t.add_argument("--sweeps", type=int, default=4)
    a = sub.add_parser("apply")
    a.add_argument("model_dir")
    a.add_argument("offsets")
    a.add_argument("out_dir")
    args = ap.parse_args()
    tune(args) if args.cmd == "tune" else apply(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/package_submission.py
"""Package a trained model folder as a competition submission.

1. Writes the model card (README.md) and LICENSE into the folder.
2. Runs the organisers' checker on the exact folder.
3. Uploads it to a private Hugging Face repo and records the commit hash.
4. Writes submission.zip with the organisers' make_submission.py.

    python src/package_submission.py <model_dir> <hf_repo> <out_dir> <rung> [--ext-manifest manifest.csv]

The repo stays private; make it public only right before uploading the zip to Codabench.
"""
import argparse
import csv
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

CARD = """---
license: cc-by-nc-sa-4.0
library_name: transformers
pipeline_tag: audio-classification
tags: [audio-classification, language-identification, nigerian-languages]
base_model: ajesujoba/AfriHuBERT
---

# Spoken language identification for 12 Nigerian languages (DSN 2026)

Submission of team "godzilla" to the DSN 2026 *Find the Language I Am Speaking* competition.

A standard `HubertForSequenceClassification` model (loads with `AutoModelForAudioClassification`, no custom code).
It predicts one of 13 labels: `yor ibo hau ego anw efi ibb idu igl nup tiv urh other`.

## Approach

{approach}

## Base model

- [ajesujoba/AfriHuBERT](https://huggingface.co/ajesujoba/AfriHuBERT), a continued pretraining of
  [utter-project/mHuBERT-147](https://huggingface.co/utter-project/mHuBERT-147) (CC BY-NC-SA 4.0). This model inherits
  that licence.

## Training data

- Competition training and validation data,
  [Data-Science-Nigeria/dsn-2026-lid-train](https://huggingface.co/datasets/Data-Science-Nigeria/dsn-2026-lid-train),
  which combines afrispeech_ipa (CC-BY-4.0), hypa_voices (Apache-2.0) and omnilingual_asr (CC-BY-4.0); see its
  ATTRIBUTION.md. No other use of those corpora.
{external}

## Usage

```python
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
fe = AutoFeatureExtractor.from_pretrained(REPO)
model = AutoModelForAudioClassification.from_pretrained(REPO)
inputs = fe(waveform_16khz_mono, sampling_rate=16000, return_tensors="pt")
label = model.config.id2label[model(**inputs).logits.argmax(-1).item()]
```
"""

APPROACH = {
    1: "AfriHuBERT truncated to its first 4 transformer layers with a weighted layer sum and mean pooling, fine-tuned on "
       "random 3-6 s crops with speaker- and source-balanced sampling. Inference uses the full clip.",
}


def external_section(manifests):
    if not manifests:
        return "- No external data."
    lines = ["", "External datasets (openly licensed; per-dataset licence shown):", ""]
    seen = set()
    for manifest in manifests:
        rows = list(csv.DictReader(open(manifest)))
        if rows and "dataset_id" in rows[0]:  # one row per dataset / config / label
            for r in rows:
                key = (r["dataset_id"], r["config_split"])
                if key in seen:
                    continue
                seen.add(key)
                lines.append(f"- [{r['dataset_id']}]({r['url']}), {r['config_split']}, revision {r['revision'][:10]}, "
                             f"{r['license']}, used as `{r['label']}`")
        else:  # one row per media file (Wikimedia Commons, Wikitongues, Bloom Library)
            by_source = {}
            for r in rows:
                by_source.setdefault(r["source"], []).append(r)
            names = {"ext_wikimedia": "Wikimedia Commons (Wikimedia Nigeria oral-history project)",
                     "ext_wikitongues": "Wikitongues (via Wikimedia Commons)", "ext_bloom": "Bloom Library talking books"}
            for src, rs in by_source.items():
                lic = sorted({r["licence"] for r in rs})
                lines.append(f"- {names.get(src, src)}: {len(rs)} recordings ({', '.join(lic)}); speech segments only, "
                             f"English and music removed:")
                for r in rs:
                    lines.append(f"  - [{r['title']}]({r['file_page']}), {r['licence']}, by {r['author'] or r['uploader']}, "
                                 f"used as `{r['label']}`")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model_dir")
    ap.add_argument("hf_repo")
    ap.add_argument("out_dir")
    ap.add_argument("rung", type=int)
    ap.add_argument("--ext-manifest", nargs="*", default=[])
    ap.add_argument("--approach")
    a = ap.parse_args()

    model_dir, out_dir = Path(a.model_dir), Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    card = CARD.format(approach=a.approach or APPROACH[a.rung], external=external_section(a.ext_manifest))
    (model_dir / "README.md").write_text(card.replace("REPO", repr(a.hf_repo)))
    shutil.copy(ROOT / "templates" / "LICENSE_CC_BY_NC_SA_4.0.txt", model_dir / "LICENSE")

    chk = subprocess.run([sys.executable, str(ROOT / "starting_kit" / "check_model_folder.py"), str(model_dir)],
                         capture_output=True, text=True)
    print(chk.stdout.strip())
    if chk.returncode != 0:
        sys.exit("checker failed: not uploading")

    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(a.hf_repo, repo_type="model", private=True, exist_ok=True)
    commit = api.upload_folder(repo_id=a.hf_repo, folder_path=str(model_dir), commit_message=f"Rung {a.rung} model")
    sha = commit.oid
    print("uploaded", a.hf_repo, sha)

    subprocess.run([sys.executable, str(ROOT / "starting_kit" / "make_submission.py"), a.hf_repo, "--revision", sha],
                   cwd=out_dir, check=True)
    (out_dir / "INFO.txt").write_text(f"repo: https://huggingface.co/{a.hf_repo}\nrevision: {sha}\nrung: {a.rung}\n")
    print("wrote", out_dir / "submission.zip")


if __name__ == "__main__":
    main()


## Exact configurations used

Each run is `python src/launch_kernel.py <kaggle_user> <slug> '<json>'`; the JSON replaces the `CONFIG` line of `src/train_kernel.py` and names the Kaggle datasets to attach.

In [ ]:
# 3-fold CV of the recipe (held-out clean/noisy/phone logits for the local leaderboard and offset tuning)
cv = {"mode": "cv", "folds": ["A", "B", "C"], "folds_file": "folds_v0_ext_va7.csv", "n_layers": 12, "steps": 8000,
      "aug": True, "eval_per_class": 200, "eval_at": [0.5, 1.0]}
# Full model 1 (D4): all data, seed 0
full1 = {"mode": "full", "folds_file": "folds_final.csv", "n_layers": 12, "steps": 8000, "batch": 32,
         "crop_min_s": 3.0, "crop_max_s": 6.0, "lr_body": 5e-05, "lr_head": 0.0005, "loss": "ce", "tau": 0.0,
         "group_alpha": 0.5, "aug": True, "ext_cap": 0.5, "seed": 0}
# Full model 2 (for the weight average): same data, same head initialisation (init_seed 0), different data order (seed 1);
# the same run also trains seed 0 / seed 1 / their average on fold A as the gate for averaging
full2 = {"mode": "cv_soup", "folds": ["A"], "folds_file": "folds_cv2.csv", "n_layers": 12, "steps": 8000, "aug": True,
         "eval_per_class": 200, "seed": 0, "soup_seeds": [0, 1], "init_seed": 0, "full_seed": 1}
# D7: the same recipe retrained with the Common Voice Ibibio and MDC Nupe sets added (folds_final3.csv)
d7 = {"mode": "full", "folds_file": "folds_final3.csv", "n_layers": 12, "steps": 8000, "aug": True, "seed": 0}


In [ ]:
# After the runs: local leaderboard, offsets, averaging, packaging
# python src/predict_lb.py splits/folds_v0_ext_va7.csv oof_A_s8000.npz oof_B_s8000.npz oof_C_s8000.npz --va va_A_s8000.npz va_B_s8000.npz va_C_s8000.npz --layers 12 --tune-out offsets_s8000.json
# python src/crossfit_offsets.py splits/folds_v0_ext_va7.csv <oof_dir> 8000 --zero nup --va      # honest check -> shrink x0.75
# python src/average_models.py d5_raw d4_full_out/model d5_soup_out/model                         # weight average
# python src/tune_bias.py apply d5_raw results/d5/offsets_d5b.json pkg_d5b                        # offsets into classifier.bias
# python starting_kit/check_model_folder.py pkg_d5b
# python src/package_submission.py pkg_d5b <hf_user>/dsn2026-lid-d5b submissions/D5b 5 --ext-manifest research/ext_manifest.csv research/voiceafrica_manifest.csv research/ext2_manifest.csv


In [ ]:
# Final per-language offsets added to classifier.bias (D5b)
offsets_d5b = {
 "yor": -0.8,
 "ibo": -2.1959,
 "hau": -0.3912,
 "ego": -0.2974,
 "anw": -2.4772,
 "efi": 2.6,
 "ibb": 2.2,
 "idu": 0.3,
 "igl": 2.1401,
 "nup": 0.4,
 "tiv": 1.1,
 "urh": -0.2974,
 "other": 0.4291
}

## Inference (what the organisers' runner does)

In [ ]:
import numpy as np, soundfile as sf, torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
REPO = "godzilla1215/dsn2026-lid-d5b"   # @ 4ea16f1e30a7045079adfc675a6f39194e58956f
fe = AutoFeatureExtractor.from_pretrained(REPO)
model = AutoModelForAudioClassification.from_pretrained(REPO).eval()
wave, sr = sf.read("clip.wav", dtype="float32")     # 16 kHz mono, 2-30 s
with torch.no_grad():
    logits = model(**fe(wave, sampling_rate=16000, return_tensors="pt")).logits
print(model.config.id2label[int(logits.argmax())])


## Results on the public leaderboard

| Submission | What | Final | Weighted F1 | Efficiency |
|---|---|---|---|---|
| D1 | 4-layer AfriHuBERT, competition data only, no augmentation | 0.3421 | 0.374 | 0.830 |
| D5 | 12 layers + augmentation + external data + cleaned labels + weight average + CV offsets | 0.6078 | 0.679 | 0.791 |
| **D5b** | D5 with offsets corrected from D5's confusion matrix | **0.6222** | 0.695 | 0.790 |
| D5c / D7 | second offset correction / retrained with Common Voice Ibibio + MDC Nupe | pending | | |

Per-language F1 of D5b: yor 0.80, ibo 0.92, hau 0.93, ego 0.96, anw 0.97, efi 0.34, ibb 0.08, idu 0.85, igl 0.35, nup 0.71, tiv 0.61, urh 0.99, other 0.82; accuracy clean 0.81 / noisy 0.82 / phone 0.78.
